In [51]:
#Import libraries
import os
import sys
sys.path.append("./scripts/")
import shutil
from brian2.units.allunits import *
from brian2.units.stdunits import *
from brian2.utils.caching import *
from scipy.sparse import coo_matrix
import numpy as np
import datalayerOmen
import random as pyrandom
import sqlalchemy as sql
import pandas as pd
from brian2 import *
from sqlalchemy import false
from sympy import true
prefs.codegen.target = "numpy"
import matplotlib.pyplot as plt
from helper import load_wmx, preprocess_monitors, generate_cue_spikes,\
                   save_vars, save_PSD, save_TFR, save_LFP, save_replay_analysis,save_wmx,save_vars_syn,SynWeightDist,save_vars_syn_cpp, _load_PF_starts 
import seaborn as sns
from sqlalchemy import create_engine
from scipy import stats
from plots import plot_violin, plot_raster, plot_posterior_trajectory, plot_PSD, plot_TFR, plot_zoomed, plot_detailed, plot_LFP, set_fig_dir, plot_wmx,set_len_sim,plot_histogram_wmx, plot_Zoom_Weights,fig_dir

In [52]:
base_path = os.path.sep.join(os.path.abspath("__file__").split(os.path.sep)[:-1])
place_cell_ratio = 0.5
PF_pklf_name = os.path.join(base_path, "files", f"PFstarts_{place_cell_ratio}_linear.pkl") if linear else None
pf_Starts = _load_PF_starts(PF_pklf_name)

In [53]:

engine = datalayerOmen.InitializeSQLEngine()
conn = engine.connect()

WARNING    /tmp/ipykernel_20294/3769643668.py:2: SAWarning: Unrecognized server version info '17.0.1115.1'.  Some SQL Server features may not function properly.
  conn = engine.connect()
 [py.warnings]


In [54]:
#Define speed using the time of the first and last spike
def speedM2 (data):
    dframe = data.copy()
    
    #dframe = dframe.sort_values (by = ['Spike_Time_int'])
    dframe = dframe.groupby (['Spike_Time_int'])
    #print(dframe)
    dframe = dframe.filter(lambda x: x['PC_ID'].count() > 2)
    dframe = dframe.groupby (['Spike_Time_int']).size().reset_index(name='count')

    #print(dframe)
    #dframe = dframe.loc(dframe['count']>=3)
   
    
    minTime = dframe['Spike_Time_int'].min()
    maxTime = dframe['Spike_Time_int'].max()
    return maxTime - minTime

In [55]:
#Cycle times calculations
def calculate_cycle_times(expid, sqlengine, hertz=0.0):
    conn = sqlengine.connect()
    SQLtext_BCs = '''SELECT [t], [BC], [Hz.]  FROM [CUNY].[dbo].[vwSpike_PC_Times - All BCs] WHERE expid = %d and [Hz.] >= %f''' % (expid, hertz)
    bcs_data = pd.read_sql(SQLtext_BCs, conn)
    conn.close()
    if bcs_data.empty:
        print("No data found for the specified expid.")
    return bcs_data

In [56]:
#Find BCs with high activity for replay duration
def find_high_activity_periods(expid, hz_threshold, bc_threshold, window_size, sqlengine):
    bcs_data = calculate_cycle_times(expid, sqlengine, hertz=hz_threshold)
    empty_periods = pd.DataFrame(columns=['Start', 'End', 'Cnt', 'Duration'])
    
    if bcs_data.empty:
        return empty_periods
    
    # Filter rows where Hz is below the threshold and keep only valid times.
    #filtered_data = bcs_data[bcs_data['Hz.'] >= hz_threshold].copy()
    filtered_data = bcs_data.dropna(subset=['t'])
    if filtered_data.empty:
        return empty_periods
    
    # Group t in buckets of 10 seconds
    filtered_data.loc[:, 't_bucket'] = (filtered_data['t'] // window_size) * window_size
    filtered_data = filtered_data.dropna(subset=['t_bucket'])
    if filtered_data.empty:
        return empty_periods
    
    # Calculate the average count of BCs per t bucket
    avg_bc_per_bucket = filtered_data.groupby('t_bucket').size().reset_index(name='BC_Count')

    min_bucket = filtered_data['t_bucket'].min()
    max_bucket = filtered_data['t_bucket'].max()
    if pd.isna(min_bucket) or pd.isna(max_bucket):
        return empty_periods
    
    min_bucket = int(min_bucket)
    max_bucket = int(max_bucket)
    all_buckets = pd.DataFrame({'t_bucket': range(min_bucket, max_bucket + window_size, window_size)})

    # Merge with avg_bc_per_bucket to include all buckets
    avg_bc_per_bucket = all_buckets.merge(avg_bc_per_bucket, on='t_bucket', how='left').fillna(0)

    #print(avg_bc_per_bucket)
    # Set initial variables
    periods = []
    start_time = None
    end_time = None
    total_bc = 0
    count = 0
    
    # Iterate over rows to find periods where average BC count is above the threshold
    for index, row in avg_bc_per_bucket.iterrows():
        if row['BC_Count'] >= bc_threshold:
            if start_time is None:
                start_time = row['t_bucket']
            end_time = row['t_bucket']  # End time is the start of the last qualifying bucket
            total_bc += row['BC_Count']
            count += 1
        else:
            if start_time is not None:
                periods.append({
                    'Start': start_time,
                    'End': end_time,
                    'Cnt': total_bc // count,
                    'Duration': end_time - start_time
                })
                start_time = None
                end_time = None
                total_bc = 0
                count = 0   
    # Handle case where the last period extends to the end of the data
    if start_time is not None:
        periods.append({
            'Start': start_time,
            'End': end_time,
            'Cnt': total_bc // count,
            'Duration': end_time - start_time
        })
    # Convert periods to DataFrame
    
    periods_df = pd.DataFrame(periods)
    
    return periods_df
    

In [57]:
#Find BCs with high activity for replay duration
def find_high_activity_periods(expid, hz_threshold, bc_threshold, window_size, sqlengine, mean_multiplier=1.5):
    bcs_data = calculate_cycle_times(expid, sqlengine, hertz=hz_threshold)
    empty_periods = pd.DataFrame(columns=['Start', 'End', 'Cnt', 'Duration'])
    
    if bcs_data.empty:
        return empty_periods
    
    # Filter rows where Hz is below the threshold and keep only valid times.
    #filtered_data = bcs_data[bcs_data['Hz.'] >= hz_threshold].copy()
    filtered_data = bcs_data.dropna(subset=['t'])
    if filtered_data.empty:
        return empty_periods
    max_time = filtered_data['t'].max()
    row_count = filtered_data.shape[0]
    avg_time_per_row = max_time / row_count if row_count > 0 else 0
    average_time_per_window = avg_time_per_row * window_size
    # Group t in buckets of 10 seconds
    filtered_data.loc[:, 't_bucket'] = (filtered_data['t'] // window_size) * window_size
    filtered_data = filtered_data.dropna(subset=['t_bucket'])
    if filtered_data.empty:
        return empty_periods
    
    # Calculate the average count of BCs per t bucket
    avg_bc_per_bucket = filtered_data.groupby('t_bucket').size().reset_index(name='BC_Count')

    min_bucket = filtered_data['t_bucket'].min()
    max_bucket = filtered_data['t_bucket'].max()
    if pd.isna(min_bucket) or pd.isna(max_bucket):
        return empty_periods
    
    min_bucket = int(min_bucket)
    max_bucket = int(max_bucket)
    all_buckets = pd.DataFrame({'t_bucket': range(min_bucket, max_bucket + window_size, window_size)})

    # Merge with avg_bc_per_bucket to include all buckets
    avg_bc_per_bucket = all_buckets.merge(avg_bc_per_bucket, on='t_bucket', how='left').fillna(0)

    #print(avg_bc_per_bucket)
    # Set initial variables
    periods = []
    start_time = None
    end_time = None
    total_bc = 0
    count = 0
    
    # Iterate over rows to find periods where average BC count is above the threshold
    for index, row in avg_bc_per_bucket.iterrows():
        #if row['BC_Count'] >= bc_threshold:
        if row['BC_Count'] >= mean_multiplier * average_time_per_window:
            if start_time is None:
                start_time = row['t_bucket']
            end_time = row['t_bucket']  # End time is the start of the last qualifying bucket
            total_bc += row['BC_Count']
            count += 1
        else:
            if start_time is not None:
                periods.append({
                    'Start': start_time,
                    'End': end_time,
                    'Cnt': total_bc // count,
                    'Duration': end_time - start_time
                })
                start_time = None
                end_time = None
                total_bc = 0
                count = 0   
    # Handle case where the last period extends to the end of the data
    if start_time is not None:
        periods.append({
            'Start': start_time,
            'End': end_time,
            'Cnt': total_bc // count,
            'Duration': end_time - start_time
        })
    # Convert periods to DataFrame
    
    periods_df = pd.DataFrame(periods)
    
    return periods_df
    

In [58]:
#Get spike data and calculate regression slope and speed
def fetch_spike_data(expid, sqlengine, hertz=0.0):
    engine = sqlengine
    conn = engine.connect()
    
    SQLtext = '''SELECT [PC_ID], Spike_Time_int, [Hz.]
                 FROM [vwSpike_PC_Times - All PCs]
                 WHERE (expid = %d) and ([Hz.] >= %f)
                 ORDER BY [Spike_Time_int]
              ''' % (expid, hertz)
    spike_data = pd.read_sql(SQLtext, conn)
    conn.close()
    
    return spike_data

def calculate_regression_slope(spike_data, expid):
    # Filter data for the specified period and PC_ID range
    try:
        if spike_data.empty:
            return None, None, None, None, None
        # Check if all x values (Spike_Time_int) are identical
        if spike_data['Spike_Time_int'].nunique() == 1:
            # Return None or some default value since regression cannot be calculated
            return None, None, None, None, None
        # Calculate the regression slope
        slope, intercept, r_value, p_value, std_err = stats.linregress(spike_data['Spike_Time_int'], spike_data['PC_ID'])
        if slope is None:
            print(f"Skipping regression due to insufficient or invalid data for experiment {expid}")
            return None, None, None, None, None  # Skip to the next experiment or handle the case as needed
        return slope, intercept, r_value, p_value, std_err
    except Exception as e:
        # Catch any other unexpected errors and return None for all outputs
        print(f"Error calculating regression: {e}")
        return None, None, None, None, None



In [71]:
#Calculate replay speeds
def replay_speeds(expid, sqlengine, hertz=0.0, min_PC_Spikes=100, mean_multiplier=1.5):
    hz_threshold = 160
    bc_threshold = 10    #Minimum number of BCs spikes in a window with high activity (Hz >= hz_threshold)
    window_size = 10    #Window size in ms in which we are looking for high activity 
    #min_PC_Spikes = 100
    
    # Fetch high activity periods and spike data
    high_activity_periods = find_high_activity_periods(expid=expid, hz_threshold=hz_threshold, bc_threshold=bc_threshold, window_size=window_size, sqlengine=sqlengine, mean_multiplier=mean_multiplier)
    if high_activity_periods.empty:
        print(f"No high activity periods found for expid {expid}")
        return pd.DataFrame()
    spike_data = fetch_spike_data(expid, sqlengine)
    if hertz is not None:
        spike_data = spike_data[spike_data['Hz.'] >= hertz]
    print(f"Found {len(high_activity_periods)} high activity periods for expid {expid}")
    Results = []
    
    for index, row in high_activity_periods.iterrows():
        start_time = row['Start']
        end_time = row['End']
        
        # Get replay data
        full_replay = spike_data[(spike_data['Spike_Time_int'] >= start_time) & (spike_data['Spike_Time_int'] < end_time)]
        
        # Check if full_replay is empty
        if full_replay.empty:
            print(f"Skipping empty replay for expid {expid}, start: {start_time}, end: {end_time}")
            continue

        # Calculate replay midpoint (median PC_ID)
        replay_midpoint = full_replay['PC_ID'].median()
        replay_section_min = replay_midpoint - 250
        replay_section_max = replay_midpoint + 250

        # Check if replay_midpoint is NaN
        if pd.isna(replay_midpoint):
            print(f"Skipping replay with NaN midpoint for expid {expid}, start: {start_time}, end: {end_time}")
            continue
        
        min_range = full_replay['PC_ID'].min()
        max_range = full_replay['PC_ID'].max()
        # Filter the replay data for the range
        replay_data = spike_data[(spike_data['Spike_Time_int'] >= start_time) & 
                                 (spike_data['Spike_Time_int'] < end_time) & 
                                 (spike_data['PC_ID'] >= replay_section_min) & 
                                 (spike_data['PC_ID'] <= replay_section_max)]
        
        # Skip if replay_data is empty after filtering
        #if replay_data.empty:
        if replay_data.shape[0] < min_PC_Spikes or replay_data.empty:
            print(f"Skipping empty filtered replay data for expid {expid}, start: {start_time}, end: {end_time}")
            continue
        
        # Calculate regression
        slope, intercept, r_value, p_value, std_err = calculate_regression_slope(replay_data, expid=expid)
        
        # Handle cases where the regression fails
        if slope is None:
            print(f"Skipping invalid regression for expid {expid}, start: {start_time}, end: {end_time}")
            continue
        
        duration = end_time - start_time
        if duration <= 10:
            print(f"Skipping short replay for expid {expid}, start: {start_time}, end: {end_time}")
            continue
        else:
            Results.append({
                'ExpID': expid,
                'Start': start_time,
                'End': end_time,
                'Slope': slope,
                'min_range': min_range,
                'max_range': max_range,
                'Duration': duration,
                'R_Value': r_value,
                'P_Value': p_value,
                'Std_Err': std_err
            })
    
    return pd.DataFrame(Results)

In [60]:
import pandas as pd
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, Float, text
from sqlalchemy.exc import OperationalError
from sqlalchemy import inspect
'''
def create_replays_table(engine):
    metadata = MetaData()
    replays_table = Table('replays', metadata,
        Column('ExpID', Integer),
        Column('Start', Float),
        Column('End', Float),
        Column('Slope', Float),
        Column('min_range', Float),
        Column('max_range', Float),
        Column('Duration', Float),
        Column('R_Value', Float),
        Column('P_Value', Float),
        Column('Std_Err', Float)
    )
    metadata.create_all(engine)
'''
def write_results_to_sql(results_df, expid, sqlengine):
    # Establish connection to the database
    engine = sqlengine
    conn = engine.connect()
    
    # Check if the table exists, and create it if it does not
    inspector = inspect(engine)
    if not inspector.has_table('replays'):
        create_replays_table(engine)
    
    trans = conn.begin()
    try:
        # Delete existing records with the same ExpID
        delete_query = f"DELETE FROM dbo.replays WHERE ExpID = {expid}"
        conn.execute(text(delete_query))

        if results_df.empty:
            trans.commit()
            print(f"No replay rows to write for expid {expid}")
            conn.close()
            return

        # Write the new results to the database
        results_df.to_sql('replays', con=conn, if_exists='append', index=False)

        # Commit the transaction
        trans.commit()
        print("Data committed successfully!")
    except Exception as e:
        # Rollback the transaction if any exception occurs
        trans.rollback()
        print(f"Transaction failed: {e}")
    # Delete existing records with the same ExpID
    #delete_query = f"DELETE FROM dbo.replays WHERE ExpID = {expid}"
    #print(delete_query)
    #conn.execute(text(delete_query))
    
    # Write the new results to the database
    #results_df.to_sql('replays', con=conn, if_exists='append', index=False)
    #print(results_df.shape[0], ' records written to SQL successfully.')
    #print('Data written to SQL successfully.')
    # Close the connection
    conn.close()





In [61]:
#create_replays_table(engine)

In [ ]:
#Process experiments for replay table
def fetch_experiment_ids(sqlengine, min_exp_id=100400, max_exp_id=110000):
    engine = sqlengine
    conn = engine.connect()
    
    SQLtext = f'''SELECT ID FROM dbo.vwExperiments where ID not in (select distinct ExpID from dbo.replays) and ID >= {min_exp_id} and ID <= {max_exp_id} order by ID'''
    experiment_ids = pd.read_sql(SQLtext, conn)
    conn.close()
    
    return experiment_ids['ID'].tolist()

def process_experiments(sqlengine, min_exp_id=100400, hertz=None, max_exp_id=110000):
    
    sql_delete_query = f"DELETE FROM dbo.replays WHERE ExpID >= {min_exp_id} AND ExpID <= {max_exp_id}"
    conn = sqlengine.connect()
    try:
        conn.execute(text(sql_delete_query))
        print(f"Deleted existing records for ExpIDs between {min_exp_id} and {max_exp_id}")
        conn.commit()
        conn.close()
    except Exception as e:
        print(f"Failed to delete existing records: {e}")
        conn.rollback()
        conn.close()
    experiment_ids = fetch_experiment_ids(sqlengine, min_exp_id=min_exp_id, max_exp_id=max_exp_id)
    for expid in experiment_ids:
        print(expid)
        results_df = replay_speeds(expid, sqlengine, hertz=hertz, min_PC_Spikes=100, mean_multiplier=2.0)
        write_results_to_sql(results_df, expid, sqlengine)

In [63]:
#process_experiments(engine, min_exp_id=108748, hertz=50.0, max_exp_id=108807)

In [64]:
# Env Replays: spike data joined with PC_Order from place_field_selected
def fetch_spike_data_with_order(expid, sqlengine, hertz=0.0):
    conn = sqlengine.connect()
    SQLtext = '''
        SELECT s.[PC_ID], s.[Spike_Time_int], pf.[PC_Order]
        FROM [vwSpike_PC_Times - All PCs] s
        inner JOIN [place_field_selected] pf
            ON s.[PC_ID] = pf.[PC_Index] AND pf.[expid] = s.[expid]
        WHERE s.[expid] = %d and s.[Hz.] >= %f
        ORDER BY s.[Spike_Time_int]
    ''' % (expid, hertz)
    spike_data = pd.read_sql(SQLtext, conn)
    conn.close()
    return spike_data

# Calculate replay speeds using PC_Order as the spatial axis
def replay_speeds_env(expid, sqlengine, hertz=0.0):
    hz_threshold = 160
    bc_threshold = 20
    window_size = 10

    high_activity_periods = find_high_activity_periods(
        expid=expid, hz_threshold=hz_threshold, bc_threshold=bc_threshold,
        window_size=window_size, sqlengine=sqlengine
    )
    if high_activity_periods.empty:
        print(f"No high activity periods found for expid {expid}")
        return pd.DataFrame()

    spike_data = fetch_spike_data_with_order(expid, sqlengine, hertz=hertz)
    if spike_data.empty:
        print(f"No spike data with PC_Order for expid {expid}")
        return pd.DataFrame()

    max_pc_order = spike_data['PC_Order'].max()
    print(f"Found {len(high_activity_periods)} high activity periods for expid {expid}, max PC_Order: {max_pc_order}")
    Results = []

    for index, row in high_activity_periods.iterrows():
        start_time = row['Start']
        end_time = row['End']

        full_replay = spike_data[
            (spike_data['Spike_Time_int'] >= start_time) &
            (spike_data['Spike_Time_int'] < end_time)
        ]
        if full_replay.empty:
            print(f"Skipping empty replay for expid {expid}, start: {start_time}, end: {end_time}")
            continue

        replay_midpoint = full_replay['PC_Order'].median()
        if pd.isna(replay_midpoint):
            print(f"Skipping replay with NaN midpoint for expid {expid}, start: {start_time}, end: {end_time}")
            continue

        #min_range = replay_midpoint - 250
        #max_range = replay_midpoint + 250
        min_range = full_replay['PC_Order'].min()
        max_range = full_replay['PC_Order'].max()

        replay_data = spike_data[
            (spike_data['Spike_Time_int'] >= start_time) &
            (spike_data['Spike_Time_int'] < end_time) &
            (spike_data['PC_Order'] >= min_range) &
            (spike_data['PC_Order'] <= max_range)
        ]
        if replay_data.empty:
            print(f"Skipping empty filtered replay data for expid {expid}, start: {start_time}, end: {end_time}")
            continue

        # Rename PC_Order to PC_ID so existing helpers (calculate_regression_slope, speedM2) work unchanged
        #replay_for_regression = replay_data.rename(columns={'PC_Order': 'PC_ID'})
        replay_for_regression = replay_data.drop(columns=['PC_ID']).rename(columns={'PC_Order': 'PC_ID'})


        slope, intercept, r_value, p_value, std_err = calculate_regression_slope(replay_for_regression)
        if slope is None:
            print(f"Skipping invalid regression for expid {expid}, start: {start_time}, end: {end_time}")
            continue

        duration = speedM2(replay_for_regression)

        Results.append({
            'ExpID': expid,
            'Start': start_time,
            'End': end_time,
            'Slope': slope,
            'min_range': min_range,
            'max_range': max_range,
            'Duration': duration,
            'R_Value': r_value,
            'P_Value': p_value,
            'Std_Err': std_err
        })

    results_df = pd.DataFrame(Results)
    if not results_df.empty and 'min_range' in results_df.columns:
        results_df = results_df[
            (results_df['min_range'] > 50) & (results_df['max_range'] < max_pc_order - 50)
        ]
    return results_df

In [65]:
# Write env replay results and process all experiments
def write_results_to_env_replays(results_df, expid, sqlengine):
    conn = sqlengine.connect()
    trans = conn.begin()
    try:
        conn.execute(text(f"DELETE FROM dbo.env_replays WHERE ExpID = {expid}"))
        if results_df.empty:
            trans.commit()
            print(f"No env replay rows to write for expid {expid}")
            conn.close()
            return
        results_df.to_sql('env_replays', con=conn, if_exists='append', index=False)
        trans.commit()
        print("Env replay data committed successfully!")
    except Exception as e:
        trans.rollback()
        print(f"Transaction failed: {e}")
    conn.close()

def fetch_experiment_ids_env(sqlengine, min_expid = None, max_expid = None):
    conn = sqlengine.connect()
    SQLtext = '''SELECT ID FROM dbo.vwExperiments
                 WHERE ID NOT IN (SELECT DISTINCT ExpID FROM dbo.env_replays)'''
    if min_expid is not None:
        SQLtext += f" and ID >= {min_expid}"
    if max_expid is not None:
        SQLtext += f" and ID <= {max_expid}"
    SQLtext += " ORDER BY ID"
    experiment_ids = pd.read_sql(SQLtext, conn)
    conn.close()
    return experiment_ids['ID'].tolist()

def process_experiments_env(sqlengine, min_expid = None, max_expid = None, hertz=0.0):
    experiment_ids = fetch_experiment_ids_env(sqlengine, min_expid, max_expid)
    for expid in experiment_ids:
        print(expid)
        results_df = replay_speeds_env(expid, sqlengine, hertz=hertz)
        write_results_to_env_replays(results_df, expid, sqlengine)

In [66]:
#process_experiments_env(engine, min_expid=103597, max_expid=109000, hertz=10.0)

In [67]:
# Cross-env replays: find replays in spikes_expid using PC_Order from pc_order_expid
def fetch_spike_data_cross_env(spikes_expid, pc_order_expid, sqlengine, hertz=0.0):
    conn = sqlengine.connect()
    SQLtext = '''
        SELECT s.[PC_ID], s.[Spike_Time_int], pf.[PC_Order]
        FROM [vwSpike_PC_Times - All PCs] s
        JOIN [place_field_selected] pf
            ON s.[PC_ID] = pf.[PC_Index] AND pf.[expid] = %d
        WHERE s.[expid] = %d and s.[Hz.] >= %f
        ORDER BY s.[Spike_Time_int]
    ''' % (pc_order_expid, spikes_expid, hertz)
    spike_data = pd.read_sql(SQLtext, conn)
    conn.close()
    return spike_data

def replay_speeds_cross_env(spikes_expid, pc_order_expid, min_slope, max_slope, sqlengine, hertz=0.0, min_duration=0.0, max_duration=15000.0, replay_min_PC_Spikes = 100):
    # Step 1: Get high activity periods from the replays table for spikes_expid
    conn = sqlengine.connect()
    replays = pd.read_sql(
        f"SELECT * FROM dbo.replays WHERE ExpID = {spikes_expid} and Duration >= {min_duration} and Duration <= {max_duration}",
        conn
    )
    conn.close()

    if replays.empty:
        print(f"No replays in table for spikes_expid {spikes_expid}")
        return pd.DataFrame()

    # Only examine periods where the same-env slope is below the minimum (cross-env candidates)
    #candidates = replays[replays['Slope'].abs() < min_slope]
    candidates = replays
    if candidates.empty:
        print(f"No cross-env candidates for spikes_expid {spikes_expid}")
        return pd.DataFrame()

    # Step 2: Fetch spike data using pc_order_expid's PC_Order
    spike_data = fetch_spike_data_cross_env(spikes_expid, pc_order_expid, sqlengine, hertz=hertz)
    if spike_data.empty:
        print(f"No spike data with PC_Order for spikes_expid {spikes_expid}, pc_order_expid {pc_order_expid}")
        return pd.DataFrame()

    max_pc_order = spike_data['PC_Order'].max()
    print(f"Found {len(candidates)} cross-env candidates for spikes_expid {spikes_expid}, max PC_Order: {max_pc_order}")
    Results = []

    for _, row in candidates.iterrows():
        start_time = row['Start']
        end_time = row['End']

        window_spikes = spike_data[
            (spike_data['Spike_Time_int'] >= start_time) &
            (spike_data['Spike_Time_int'] < end_time)
        ]
        if window_spikes.empty:
            continue

        replay_midpoint = window_spikes['PC_Order'].median()
        if pd.isna(replay_midpoint):
            continue

        min_range_spikes = replay_midpoint - 250
        max_range_spikes = replay_midpoint + 250
        min_range = window_spikes['PC_Order'].min()
        max_range = window_spikes['PC_Order'].max()
        replay_data = window_spikes[
            (window_spikes['PC_Order'] >= min_range_spikes) &
            (window_spikes['PC_Order'] <= max_range_spikes)
        ]
        if replay_data.empty or replay_data.shape[0] < replay_min_PC_Spikes:
            continue

        replay_for_regression = replay_data.drop(columns=['PC_ID']).rename(columns={'PC_Order': 'PC_ID'})

        slope, intercept, r_value, p_value, std_err = calculate_regression_slope(replay_for_regression, expid=spikes_expid)
        if slope is None:
            continue

        if abs(slope) < min_slope or abs(slope) > max_slope:
            continue

        #duration = speedM2(replay_for_regression)
        duration = end_time - start_time  # Alternatively, use the duration from the replays table if more appropriate

        Results.append({
            'SpikesExpID': spikes_expid,
            'PCOrderExpID': pc_order_expid,
            'Start': start_time,
            'End': end_time,
            'Slope': slope,
            'min_range': min_range,
            'max_range': max_range,
            'Duration': duration,
            'R_Value': r_value,
            'P_Value': p_value,
            'Std_Err': std_err
        })

    results_df = pd.DataFrame(Results)
    
    return results_df

In [68]:
# Write cross-env replay results and process all experiments
def write_results_to_cross_env_replays(results_df, spikes_expid, sqlengine):
    conn = sqlengine.connect()
    trans = conn.begin()
    try:
        conn.execute(text(
            f"DELETE FROM dbo.cross_env_replays WHERE SpikesExpID = {spikes_expid}"
        ))
        if results_df.empty:
            trans.commit()
            print(f"No cross-env replay rows to write for spikes_expid {spikes_expid}")
            conn.close()
            return
        results_df.to_sql('cross_env_replays', con=conn, if_exists='append', index=False)
        trans.commit()
        print("Cross-env replay data committed successfully!")
    except Exception as e:
        trans.rollback()
        print(f"Transaction failed: {e}")
    conn.close()

def fetch_experiment_ids_cross_env(pc_order_expid, sqlengine, min_expid=None, max_expid=None):
    conn = sqlengine.connect()
    SQLtext = f'''SELECT ID FROM dbo.vwExperiments
                 WHERE ID NOT IN (
                     SELECT DISTINCT SpikesExpID FROM dbo.cross_env_replays
                     WHERE PCOrderExpID = {pc_order_expid}
                 )'''
    if min_expid is not None:
        SQLtext += f" AND ID >= {min_expid}"
    if max_expid is not None:
        SQLtext += f" AND ID <= {max_expid}"
    SQLtext += " ORDER BY ID"
    experiment_ids = pd.read_sql(SQLtext, conn)
    conn.close()
    return experiment_ids['ID'].tolist()

def get_previous_experiment_ids(spikes_expid, lookback, sqlengine):
    conn = sqlengine.connect()
    exp_ids = pd.read_sql(
        f"SELECT TOP {lookback} ID FROM dbo.vwExperiments WHERE ID <= {spikes_expid} ORDER BY ID DESC",
        conn
    )
    conn.close()
    return exp_ids['ID'].tolist()

def fetch_processed_pairs(sqlengine):
    conn = sqlengine.connect()
    pairs = pd.read_sql(
        "SELECT DISTINCT SpikesExpID, PCOrderExpID FROM dbo.cross_env_replays",
        conn
    )
    conn.close()
    return set(zip(pairs['SpikesExpID'], pairs['PCOrderExpID']))

def process_experiments_cross_env(lookback, min_slope, max_slope, sqlengine, min_expid=None, max_expid=None, hertz=0.0, min_duration=0.0, max_duration=15000.0):
    conn = sqlengine.connect()
    SQLtext = "SELECT ID FROM dbo.vwExperiments WHERE 1=1"
    if min_expid is not None:
        SQLtext += f" AND ID >= {min_expid}"
    if max_expid is not None:
        SQLtext += f" AND ID <= {max_expid}"
    SQLtext += " ORDER BY ID"
    experiment_ids = pd.read_sql(SQLtext, conn)['ID'].tolist()
    conn.close()

    for spikes_expid in experiment_ids:
        prev_exp_ids = get_previous_experiment_ids(spikes_expid, lookback, sqlengine)
        all_results = []
        for pc_order_expid in prev_exp_ids:
            print(f"{spikes_expid} -> pc_order: {pc_order_expid}")
            results_df = replay_speeds_cross_env(spikes_expid, pc_order_expid, min_slope, max_slope, sqlengine, hertz=hertz, min_duration=min_duration, max_duration=max_duration)
            if not results_df.empty:
                all_results.append(results_df)

        if all_results:
            combined = pd.concat(all_results, ignore_index=True)
            # One row per replay event: keep the best-matching environment (highest R_value)
            best_idx = combined.groupby(['Start', 'End'])['R_Value'].idxmax()
            best_matches = combined.loc[best_idx].reset_index(drop=True)
        else:
            best_matches = pd.DataFrame()

        write_results_to_cross_env_replays(best_matches, spikes_expid, sqlengine)


In [69]:
process_experiments_cross_env(
    lookback=7,
    min_slope=1.5,
    max_slope=20.0,
    sqlengine=engine,
    min_expid=108748,
    max_expid=109000,
    hertz=40.0,
    min_duration=200.0,
    max_duration=1200.0
)   


108748 -> pc_order: 108748
Found 14 cross-env candidates for spikes_expid 108748, max PC_Order: 3999
108748 -> pc_order: 108747
Found 14 cross-env candidates for spikes_expid 108748, max PC_Order: 3999
108748 -> pc_order: 108746
Found 14 cross-env candidates for spikes_expid 108748, max PC_Order: 3999
108748 -> pc_order: 108745
Found 14 cross-env candidates for spikes_expid 108748, max PC_Order: 3999
108748 -> pc_order: 108744
Found 14 cross-env candidates for spikes_expid 108748, max PC_Order: 3999
108748 -> pc_order: 108743
Found 14 cross-env candidates for spikes_expid 108748, max PC_Order: 3999
108748 -> pc_order: 108742
Found 14 cross-env candidates for spikes_expid 108748, max PC_Order: 3999
Cross-env replay data committed successfully!
108749 -> pc_order: 108749
Found 18 cross-env candidates for spikes_expid 108749, max PC_Order: 3999
108749 -> pc_order: 108748
Found 18 cross-env candidates for spikes_expid 108749, max PC_Order: 3999
108749 -> pc_order: 108747
Found 18 cross-env

In [20]:
#Code for environments
sql = '''UPDATE  cross_env_replays
SET              Env_Name = ep.env
FROM         cross_env_replays INNER JOIN
                         [Experiments Parameters] AS ep ON cross_env_replays.PCOrderExpID = ep.ExpID'''
conn = engine.connect()
conn.execute(text(sql))
conn.close()

In [122]:
#Parse the experiment parameters from the JSON and write them to SQL
import pandas as pd
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, Float, String, inspect

def fetch_experiment_params(sqlengine):
    engine = sqlengine
    conn = engine.connect()
    
    SQLtext = '''SELECT ID, [Param JSON] FROM vwExperimentsAll'''
    experiment_params = pd.read_sql(SQLtext, conn)
    conn.close()
    
    return experiment_params

def process_json_params(experiment_params):
    data = []

    for _, row in experiment_params.iterrows():
        expid = row['ID']
        json_param = row['Param JSON']
        params = json_param.split(', ')
        
        for param in params:
            if '=' in param:
                key, value = param.split('=', 1)
                key = key.strip().replace(" ", "_").lower()
                value = value.strip().replace(" ms", "")
                if value in ["True", "False"]:
                    value = value == "True"
                else:
                    try:
                        value = float(value)
                    except ValueError:
                        pass
                data.append({'ExpID': expid, 'Attribute': key, 'Value': value})
    
    data_df = pd.DataFrame(data)
    pivot_df = data_df.pivot(index='ExpID', columns='Attribute', values='Value').reset_index()
    pivot_df.columns.name = None

    # Coerce each column to numeric where possible so dtypes are float64/int64 not object
    for col in pivot_df.columns:
        if col == 'ExpID':
            continue
        coerced = pd.to_numeric(pivot_df[col], errors='coerce')
        if coerced.notna().any():
            pivot_df[col] = coerced


    return pivot_df

def _sql_col_type(series):
    non_null = series.dropna()
    if pd.api.types.is_bool_dtype(series):
        return Integer
    if pd.api.types.is_float_dtype(series):
        if len(non_null) > 0 and (non_null % 1 == 0).all():
            return Integer
        return Float
    if pd.api.types.is_integer_dtype(series):
        return Integer
    return String

def create_experiments_parameters_table(engine, df):
    metadata = MetaData()
    table_name = 'Experiments Parameters'
    
    inspector = inspect(engine)
    if inspector.has_table(table_name):
        Table(table_name, metadata).drop(engine)
    
    table_columns = [Column('ExpID', Integer)]
    for column in df.columns:
        if column == 'ExpID':
            continue
        table_columns.append(Column(column, _sql_col_type(df[column])))
        
    experiments_parameters_table = Table(table_name, metadata, *table_columns, extend_existing=True)
    metadata.create_all(engine)

def write_params_to_sql(processed_df, sqlengine):
    engine = sqlengine
    conn = engine.connect()
    create_experiments_parameters_table(engine, processed_df)
    processed_df.to_sql('Experiments Parameters', con=conn, if_exists='append', index=False)
    conn.close()

# Example usage

In [123]:
#Update the Exp paramenters table
experiment_params = fetch_experiment_params(engine)
processed_df = process_json_params(experiment_params)
#processed_df['post_AuC'] = processed_df.apply(lambda row: abs(calculate_area_under_curve(row['am'], row['taum'])), axis=1)
#processed_df['pre_AuC'] = processed_df.apply(lambda row: abs(calculate_area_under_curve(row['ap'], row['taup'],start=0,end=150)), axis=1)
write_params_to_sql(processed_df, engine)
print(processed_df)

       ExpID  adaptation_mult     am  am_bc_e  am_bc_i  am_ca1_bc  am_ca1_pc  \
0          3              1.0 -0.007      NaN      NaN        NaN        NaN   
1     100571              1.0  0.018      0.0      0.0        NaN        NaN   
2     100572              1.0  0.016      0.0      0.0        NaN        NaN   
3     100573              1.0  0.016      0.0      0.0        NaN        NaN   
4     100574              1.0  0.018      0.0      0.0        NaN        NaN   
...      ...              ...    ...      ...      ...        ...        ...   
2942  108803              1.0  0.001      0.0      0.0        NaN        NaN   
2943  108804              1.0  0.001      0.0      0.0        NaN        NaN   
2944  108805              1.0  0.001      0.0      0.0        NaN        NaN   
2945  108806              1.0  0.001      0.0      0.0        NaN        NaN   
2946  108807              1.0  0.000      0.0      0.0        NaN        NaN   

      am_pc_i     ap  ap_bc_e  ...  tau

In [17]:
# retrieve all experiments from experiment parameters table. Just expid and duration
def fetch_experiment_ids_and_duration(sqlengine, min_expid, max_expid):
    engine = sqlengine
    conn = engine.connect()
    
    SQLtext = '''SELECT expid, total_duration FROM dbo.[Experiments Parameters] where expid >= %d and expid <= %d''' % (min_expid, max_expid)
    experiment_ids = pd.read_sql(SQLtext, conn)
    conn.close()
    
    return experiment_ids


In [18]:
from sqlalchemy import text

def process_experiment_end_section(sqlengine, min_expid, max_expid):
    experiment_ids = fetch_experiment_ids_and_duration(sqlengine, min_expid, max_expid)
    batch_size = 25

    # Use a transaction context; it will commit on success or rollback on error
    with sqlengine.begin() as conn:
        # Delete existing rows for the range
        conn.execute(
            text("""
                DELETE FROM dbo.replays_end_section
                WHERE ExpID >= :min_expid AND ExpID <= :max_expid
            """),
            {"min_expid": min_expid, "max_expid": max_expid}
        )

        results_list = []

        for _, row in experiment_ids.iterrows():
            duration_start = float(row['total_duration']) - 3000
            duration_end = float(row['total_duration'])

            sqlquery = f"""
                SELECT * FROM dbo.spikedata
                WHERE expid = {row['expid']}
                AND spike_time >= {duration_start}
                AND previous_spike_time > 0
            """
            spike_data = pd.read_sql(sqlquery, conn)

            sqlquerybc = f"""
                SELECT * FROM dbo.BCs
                WHERE expid = {row['expid']}
                AND t >= {duration_start}
                AND previous_spike_time > 0
                AND 1000 / (t - previous_spike_time) >= 120
            """
            bc_spike_data = pd.read_sql(sqlquerybc, conn)

            spike_data = spike_data.sort_values(by=['Spike_Time'])
            bc_spike_data = bc_spike_data.sort_values(by=['t'])

            while duration_start < duration_end:
                batch_spike_data = spike_data[
                    (spike_data['Spike_Time'] >= duration_start) &
                    (spike_data['Spike_Time'] < duration_start + batch_size)
                ]
                batch_bc_spike_data = bc_spike_data[
                    (bc_spike_data['t'] >= duration_start) &
                    (bc_spike_data['t'] < duration_start + batch_size)
                ]

                if not batch_spike_data.empty:
                    if batch_spike_data['Spike_Time'].nunique() == 1:
                        print(
                            f"Skipping regression for ExpID {row['expid']}, "
                            f"Start {duration_start}: All spike times are identical"
                        )
                    else:
                        slope, intercept, r_value, p_value, std_err = stats.linregress(
                            batch_spike_data['Spike_Time'],
                            batch_spike_data['PC_ID']
                        )

                        results_list.append({
                            'ExpID': row['expid'],
                            'Start': duration_start,
                            'End': duration_start + batch_size,
                            'Slope': slope,
                            'BCs cnt': batch_bc_spike_data.shape[0],
                            'MinPC': batch_spike_data['PC_ID'].min(),
                            'MaxPC': batch_spike_data['PC_ID'].max()
                        })

                duration_start += batch_size

            print(f"Experiment {row['expid']} done")

        results = pd.DataFrame(results_list)
        if not results.empty:
            # This append is inside the same transaction as the delete
            results.to_sql('replays_end_section', con=conn, if_exists='append', index=False)
            print("Data committed successfully!")


In [ ]:
#process_experiment_end_section(engine, 100653, 200000)

In [32]:
def process_experiment_end_section_env(sqlengine, min_expid, max_expid):
    experiment_ids = fetch_experiment_ids_and_duration(sqlengine, min_expid, max_expid)
    batch_size = 25

    with sqlengine.begin() as conn:
        conn.execute(
            text("""
                DELETE FROM dbo.replays_end_section_env
                WHERE ExpID >= :min_expid AND ExpID <= :max_expid
            """),
            {"min_expid": min_expid, "max_expid": max_expid}
        )

        results_list = []

        for _, row in experiment_ids.iterrows():
            duration_start = float(row['total_duration']) - 3000
            duration_end = float(row['total_duration'])

            sqlquery = f"""
                SELECT s.[PC_ID], s.[Spike_Time], s.[previous_spike_time], pf.[PC_Order]
                FROM dbo.spikedata s
                JOIN dbo.place_field_selected pf
                    ON s.[PC_ID] = pf.[PC_Index] AND pf.[expid] = s.[expid]
                WHERE s.[expid] = {row['expid']}
                AND s.[spike_time] >= {duration_start}
                AND s.[previous_spike_time] > 0
            """
            spike_data = pd.read_sql(sqlquery, conn)

            sqlquerybc = f"""
                SELECT * FROM dbo.BCs
                WHERE expid = {row['expid']}
                AND t >= {duration_start}
                AND previous_spike_time > 0
                AND 1000 / (t - previous_spike_time) >= 120
            """
            bc_spike_data = pd.read_sql(sqlquerybc, conn)

            if spike_data.empty:
                print(f"No spike data with PC_Order for ExpID {row['expid']}, skipping")
                continue

            spike_data = spike_data.sort_values(by=['Spike_Time'])
            bc_spike_data = bc_spike_data.sort_values(by=['t'])

            while duration_start < duration_end:
                batch_spike_data = spike_data[
                    (spike_data['Spike_Time'] >= duration_start) &
                    (spike_data['Spike_Time'] < duration_start + batch_size)
                ]
                batch_bc_spike_data = bc_spike_data[
                    (bc_spike_data['t'] >= duration_start) &
                    (bc_spike_data['t'] < duration_start + batch_size)
                ]

                if not batch_spike_data.empty:
                    if batch_spike_data['Spike_Time'].nunique() == 1:
                        print(
                            f"Skipping regression for ExpID {row['expid']}, "
                            f"Start {duration_start}: All spike times are identical"
                        )
                    else:
                        slope, intercept, r_value, p_value, std_err = stats.linregress(
                            batch_spike_data['Spike_Time'],
                            batch_spike_data['PC_Order']
                        )

                        results_list.append({
                            'ExpID': row['expid'],
                            'Start': duration_start,
                            'End': duration_start + batch_size,
                            'Slope': slope,
                            'BCs cnt': batch_bc_spike_data.shape[0],
                            'MinPC': batch_spike_data['PC_Order'].min(),
                            'MaxPC': batch_spike_data['PC_Order'].max()
                        })

                duration_start += batch_size

            print(f"Experiment {row['expid']} done")

        results = pd.DataFrame(results_list)
        if not results.empty:
            results.to_sql('replays_end_section_env', con=conn, if_exists='append', index=False)
            print("Data committed successfully!")

In [ ]:
#process_experiment_end_section_env(engine, 103591, 200000)